<a href="https://colab.research.google.com/github/Vipul-tyagi/catalogiq/blob/main/Amazon_category_wise_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
#Cell1
from google.colab import drive
import duckdb

# 1. Mount Google Drive
print("Mounting Google Drive...")
drive.mount('/content/drive')
print("Google Drive mounted successfully.")

# 2. Install the 'duckdb' library
print("Installing 'duckdb' library...")
!pip install duckdb
print("'duckdb' installed successfully.")

# 3. Connect to an in-memory DuckDB database and execute SQL commands
print("Connecting to DuckDB and configuring httpfs...")
con = duckdb.connect(database=':memory:', read_only=False)
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")
print("DuckDB configured with httpfs.")

# Print a success message
print("\nEnvironment Ready!")

Mounting Google Drive...
Mounted at /content/drive
Google Drive mounted successfully.
Installing 'duckdb' library...
'duckdb' installed successfully.
Connecting to DuckDB and configuring httpfs...
DuckDB configured with httpfs.

Environment Ready!


In [18]:

#Cell2
# 1. Mount Google Drive for permanent storage
from google.colab import drive
import os

print("Requesting Google Drive access...")
drive.mount('/content/drive')

# Create a dedicated folder for our daily outputs if it doesn't exist
output_dir = '/content/drive/MyDrive/Amazon_Review_Insights'
os.makedirs(output_dir, exist_ok=True)
print(f"Storage routed to: {output_dir}")

# 2. Securely load the Gemini API Key
from google.colab import userdata
import google.generativeai as genai

print("Loading secure credentials...")
api_key = userdata.get('Gemini_API_check')
genai.configure(api_key=api_key)
print("Gemini API authenticated successfully.")

# 3. Install and configure DuckDB for remote HTTP streaming
print("Installing DuckDB streaming engine...")
!pip install duckdb --quiet
import duckdb

# Connect to an in-memory database and load the web-streaming extensions
con = duckdb.connect(database=':memory:')
con.execute("INSTALL httpfs; LOAD httpfs;")
print("DuckDB engine active and ready to stream Hugging Face data.")

print("\n✅ PHASE 1 COMPLETE: Environment is fully activated.")

Requesting Google Drive access...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Storage routed to: /content/drive/MyDrive/Amazon_Review_Insights


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Loading secure credentials...
Gemini API authenticated successfully.
Installing DuckDB streaming engine...
DuckDB engine active and ready to stream Hugging Face data.

✅ PHASE 1 COMPLETE: Environment is fully activated.


In [19]:
#Cell3
import os
import duckdb
import pandas as pd
from google.colab import userdata

print("📥 Bypassing broken Hugging Face libraries. Initiating direct raw download...")

hf_token = userdata.get('HF_TOKEN')
category = "Office_Products"

# The true raw CDN endpoints (These are plain .jsonl files)
meta_url = f"https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw/meta_categories/meta_{category}.jsonl"
review_url = f"https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw/review_categories/{category}.jsonl"

# 1. Force the download directly to Colab's SSD using the Linux command line
print(f"Downloading {category} Metadata...")
os.system(f"wget -q --header='Authorization: Bearer {hf_token}' -O meta.jsonl {meta_url}")

print(f"Downloading {category} Reviews (This is a multi-gigabyte file, please wait)...")
os.system(f"wget -q --header='Authorization: Bearer {hf_token}' -O reviews.jsonl {review_url}")

print("✅ Files secured locally on Colab SSD. Launching DuckDB engine...")

# 2. Connect to DuckDB locally
con = duckdb.connect(database=':memory:')

# 3. Query the local JSONL files directly
query = """
WITH TargetSubCategories AS (
    SELECT categories[2] as sub_category, COUNT(*) as total_reviews
    FROM read_json_auto('meta.jsonl') m
    WHERE len(categories) >= 2 AND categories[2] IS NOT NULL
    GROUP BY sub_category
    ORDER BY total_reviews DESC
    LIMIT 5
),
TopSKUs AS (
    SELECT
        m.categories[2] as sub_category,
        m.parent_asin,
        m.title,
        m.store as brand,
        COUNT(r.rating) as review_count,
        ROW_NUMBER() OVER(PARTITION BY m.categories[2] ORDER BY COUNT(r.rating) DESC) as sku_rank
    FROM read_json_auto('meta.jsonl') m
    JOIN read_json_auto('reviews.jsonl') r ON m.parent_asin = r.parent_asin
    WHERE m.categories[2] IN (SELECT sub_category FROM TargetSubCategories)
      AND m.title IS NOT NULL
    GROUP BY m.categories[2], m.parent_asin, m.title, m.store
    HAVING COUNT(r.rating) > 50
),
RankedReviews AS (
    SELECT
        r.parent_asin,
        r.rating,
        r.text,
        ROW_NUMBER() OVER(PARTITION BY r.parent_asin ORDER BY r.helpful_vote DESC) as review_rank
    FROM read_json_auto('reviews.jsonl') r
    WHERE r.parent_asin IN (SELECT parent_asin FROM TopSKUs WHERE sku_rank <= 20)
      AND r.text IS NOT NULL
)
SELECT
    t.sub_category, t.parent_asin, t.title, t.brand, t.review_count,
    r.rating, r.text
FROM TopSKUs t
JOIN RankedReviews r ON t.parent_asin = r.parent_asin
WHERE t.sku_rank <= 20 AND r.review_rank <= 50
ORDER BY t.sub_category, t.parent_asin, r.review_rank;
"""

print(f"Aggregating Top SKUs and consumer sentiment matrices. This will take 1-3 minutes...")
pipeline_df = con.execute(query).df()

print(f"\n✅ Extraction Complete: {len(pipeline_df)} high-density reviews pulled for {pipeline_df['parent_asin'].nunique()} Top SKUs.")
print("\n--- Preview of Extracted Payload ---")
print(pipeline_df[['sub_category', 'title', 'rating']].head(5))

📥 Bypassing broken Hugging Face libraries. Initiating direct raw download...
✅ Files secured locally on Colab SSD. Launching DuckDB engine...
Aggregating Top SKUs and consumer sentiment matrices. This will take 1-3 minutes...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


✅ Extraction Complete: 4200 high-density reviews pulled for 84 Top SKUs.

--- Preview of Extracted Payload ---
      sub_category                                              title  rating
0  Education Store  Carson Dellosa Education Carson Dellosa Differ...     5.0
1  Education Store  Carson Dellosa Education Carson Dellosa Differ...     3.0
2  Education Store  Carson Dellosa Education Carson Dellosa Differ...     5.0
3  Education Store  Carson Dellosa Education Carson Dellosa Differ...     2.0
4  Education Store  Carson Dellosa Education Carson Dellosa Differ...     4.0


In [20]:
import requests
import json
import time
from google.colab import userdata

print("🧠 Initiating Phase 3: Amazon Category Intelligence Framework...")

# 1. Retrieve API Key
api_key = userdata.get('Gemini_API_check')
url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-flash:generateContent?key={api_key}"

# 2. Package the reviews
print("Packaging pipeline_df into the Mega-Prompt...")
payload_data = pipeline_df.to_json(orient="records")

# 3. Inject the Enterprise Mega-Prompt
system_instruction = """
Amazon Category Intelligence Prompt
You are a team of Senior B2B Ecommerce Product Managers, Category Managers, Consumer Research Analysts, and Data Scientists.

You are analyzing actual Amazon Reviews 2023 review and metadata data. Do NOT generate synthetic insights. Do NOT speculate.
The objective is to understand what buyers care about, purchase drivers, objections, metadata gaps, and B2B friction points.

Return ONLY valid JSON matching the following structure exactly:
{
  "category_analysis": {
    "category_name": "string",
    "products_analyzed": 0,
    "reviews_analyzed": 0,
    "top_buyer_concerns": [{"concern": "string", "frequency_percent": 0, "confidence": "high|medium|low"}],
    "top_purchase_drivers": [{"driver": "string", "frequency_percent": 0, "confidence": "high|medium|low"}],
    "top_objections": [{"objection": "string", "frequency_percent": 0, "confidence": "high|medium|low"}],
    "helpful_review_themes": [{"theme": "string", "average_helpful_votes": 0}],
    "metadata_enrichment_opportunities": [{"attribute": "string", "review_frequency_percent": 0, "listing_coverage_percent": 0, "gap_score": 0}],
    "winner_attributes": [{"attribute": "string", "top100_frequency": 0, "category_frequency": 0, "lift": 0}],
    "b2b_friction_points": [{"issue": "string", "evidence_strength": "high|medium|low"}],
    "brand_insights": [{"brand": "string", "strength": "string", "weakness": "string", "buyer_association": "string"}],
    "subcategory_breakdowns": [{"subcategory_name": "string", "primary_purchase_driver": "string", "top_complaint": "string", "top_buyer_concern": "string", "metadata_gap": "string", "winner_attribute": "string", "sku_specific_callout": "string"}],
    "top_strategic_findings": [{"finding": "string", "supporting_evidence": "string", "confidence": "high|medium|low"}]
  }
}
"""

request_body = {
    "system_instruction": {"parts": [{"text": system_instruction}]},
    "contents": [{"parts": [{"text": f"Analyze this data:\\n{payload_data}"}]}],
    "generationConfig": {"temperature": 0.2, "responseMimeType": "application/json"}
}

print("🚀 Transmitting payload to Gemini...")
start_time = time.time()
response = requests.post(url, headers={'Content-Type': 'application/json'}, json=request_body)

if response.status_code == 200:
    print(f"✅ Analysis Complete in {round(time.time() - start_time, 2)} seconds!")
    insights = json.loads(response.json()['candidates'][0]['content']['parts'][0]['text'])
    final_json_output = insights
else:
    print(f"❌ API Call Failed: {response.status_code}\\n{response.text}")

🧠 Initiating Phase 3: Direct HTTP API Bypass (Using Gemini 3.5 Flash)...
Packaging reviews into the Mega-Prompt...
🚀 Transmitting 1M-token payload directly to Gemini servers. Analyzing sentiment...
✅ Analysis Complete in 45.1 seconds!

--- 📊 PREVIEW OF AI INSIGHTS ---
{
  "category_analysis": {
    "top_friction_points": [
      "Severe chemical off-gassing and toxic odors from rubber, foam, and plastic materials (e.g., mouse pads, desk mats) that cause headaches and transfer smells to hands.",
      "Mechanical and structural failures under normal load, such as plastic gears stripping in shredders, plastic joints snapping on rolling carts, and office chair wheels breaking base sockets.",
      "Deceptive packaging and quantity descriptions, where multi-packs or bundles are delivered with missing components or misleading unit counts.",
      "Dye transfer from seat cushions permanently staining expensive leather office chairs, and memory foam flattening instantly under average adult we

In [28]:
#Cell5
import os
import json
from google.colab import drive

# 1. Mount Google Drive
print("Mounting Google Drive...")
drive.mount('/content/drive')

# 2. Define your permanent storage path
OUTPUT_DIR = '/content/drive/MyDrive/Amazon_Review_Insights'
MASTER_INSIGHTS_PATH = os.path.join(OUTPUT_DIR, 'b2b_master_catalog_insights.json')

def append_to_master_insights(category_key, new_analysis_data):
    """Safely appends or updates category insights in the master JSON file."""

    # Ensure the directory exists
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # Load existing data if the file already exists
    master_data = {}
    if os.path.exists(MASTER_INSIGHTS_PATH):
        with open(MASTER_INSIGHTS_PATH, 'r') as f:
            try:
                master_data = json.load(f)
            except json.JSONDecodeError:
                print("Warning: Existing file was empty or corrupted. Starting fresh.")
                pass

    # Extract the core insights (handling nested schemas)
    if 'category_analysis' in new_analysis_data:
        master_data[category_key] = new_analysis_data['category_analysis']
    else:
        master_data[category_key] = new_analysis_data

    # Save the consolidated dictionary back to Google Drive
    with open(MASTER_INSIGHTS_PATH, 'w') as f:
        json.dump(master_data, f, indent=2)

    print(f"✅ Successfully saved '{category_key}' to Master Catalog File!")
    print(f"📂 Current Tracked Categories: {list(master_data.keys())}")

# --- EXECUTE THE APPEND ---
# Assuming 'final_json_output' is still in memory from your Phase 3 extraction
append_to_master_insights("Office_Products", final_json_output)

Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Successfully saved 'Office_Products' to Master Catalog File!
📂 Current Tracked Categories: ['Office_Products']


In [33]:
import requests
import json
import duckdb
import os
from google.colab import userdata

# Retrieve API Key and target the strict REST endpoint
api_key = userdata.get('Gemini_API_check')
url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-flash:generateContent?key={api_key}"

def ask_master_agent(question):
    if 'pipeline_df' not in globals():
        return "❌ Data Missing: Please re-run Cell 3 to restore 'pipeline_df' to global RAM."

    global_df = globals()['pipeline_df']

    # Load master strategic matrix from Drive
    MASTER_INSIGHTS_PATH = '/content/drive/MyDrive/Amazon_Review_Insights/b2b_master_catalog_insights.json'
    strategic_data = "{}"
    if os.path.exists(MASTER_INSIGHTS_PATH):
        with open(MASTER_INSIGHTS_PATH, 'r') as f:
            strategic_data = f.read()

    system_instruction = f"""
    You are an expert E-Commerce Data Agent. You have access to two data sources:
    SOURCE A: A DuckDB table named 'pipeline_df' containing raw user reviews and ratings.
    SOURCE B: Master Strategic Insights JSON spanning multiple catalog categories: {strategic_data}

    ROUTING LOGIC:
    - If the user asks a qualitative question (e.g., "What are the core objections or brand insights for Office Products?"), synthesize an elegant response directly from SOURCE B.
    - If the user asks a quantitative query (e.g., "Find the count of reviews grouped by brand"), return ONLY a valid raw DuckDB SQL query targeting 'pipeline_df'. No markdown tags.
    """

    request_body = {
        "system_instruction": {"parts": [{"text": system_instruction}]},
        "contents": [{"parts": [{"text": f"User Question: {question}"}]}],
        "generationConfig": {"temperature": 0.1}
    }

    try:
        response = requests.post(url, headers={'Content-Type': 'application/json'}, json=request_body)
        ai_response = response.json()['candidates'][0]['content']['parts'][0]['text'].strip()

        if ai_response.lower().startswith("select") or ai_response.lower().startswith("with"):
            print("🔍 Executing Live Database Query on pipeline_df...\n")
            local_con = duckdb.connect()
            local_con.register('pipeline_df', global_df)
            return local_con.execute(ai_response).df()
        else:
            print("🧠 Retrieving from Google Drive Strategic Insights...\n")
            return ai_response
    except Exception as e:
        return f"❌ Agent Error: {e}"

# --- ACTIVE VALIDATION ---
print("--- Test 1: Strategic Long-Term Memory ---")
print(ask_master_agent("What are the key brand insights found for Belkin or HP?"))

print("\n--- Test 2: Live Relational Query ---")
print(ask_master_agent("SELECT brand, COUNT(*) as reviews FROM pipeline_df GROUP BY brand ORDER BY reviews DESC LIMIT 3;"))

print("\n--- Test 3: Live Relational Query ---")
print(ask_master_agent(" which brands have the worst reviews?"))

print("\n--- Test 4: Live Relational Query ---")
print(ask_master_agent("which ergonomic desk is highly rated across brands? give me exact metadata"))

--- Test 1: Strategic Long-Term Memory ---
🧠 Retrieving from Google Drive Strategic Insights...

Based on the strategic insights for the **Office Products** category, here are the key brand insights for **HP** (analyzed alongside Brother) and **Belkin**:

### **HP / Brother**
* **Strength:** Delivers high-quality print output and highly reliable OEM hardware performance.
* **Weakness:** Implements aggressive firmware locks that block third-party ink/toner cartridges and triggers premature low-toner warnings.
* **Buyer Association:** Viewed by customers as an expensive but necessary business infrastructure.

### **Belkin**
* **Strength:** Offers low-friction tracking surfaces and comfortable gel wrist supports.
* **Weakness:** Suffers from extremely strong, toxic, and persistent chemical or petroleum odors out of the box.
* **Buyer Association:** Regarded as functional but highly odorous desk accessories.

--- Test 2: Live Relational Query ---
🔍 Executing Live Database Query on pipeline

In [29]:
#Cell7
# @title 📊 Enterprise Category Intelligence Hub {display-mode: "form"}
import json
import os
from IPython.display import HTML
from google.colab import output

def _report_js_error(message):
    print(f"JavaScript Error: {message}")
output.register_callback('report_js_error', _report_js_error)

MASTER_INSIGHTS_PATH = '/content/drive/MyDrive/Amazon_Review_Insights/b2b_master_catalog_insights.json'

master_data = {}
if os.path.exists(MASTER_INSIGHTS_PATH):
    with open(MASTER_INSIGHTS_PATH, 'r') as f:
        try: master_data = json.load(f)
        except json.JSONDecodeError: pass

if not master_data and 'final_json_output' in globals():
    master_data = {"Office_Products": final_json_output.get('category_analysis', final_json_output)}

json_payload = json.dumps(master_data)

html_content = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <link href="https://cdn.jsdelivr.net/npm/tailwindcss@2.2.19/dist/tailwind.min.css" rel="stylesheet">
    <style>
        body { background-color: #f4f6f8; font-family: ui-sans-serif, system-ui, -apple-system, sans-serif; }
        .card { background: white; border-radius: 12px; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1); padding: 1.5rem; }
        .tab-active { border-bottom-width: 2px; border-color: #2563eb; font-weight: 700; color: #2563eb; }
        .tab-inactive { color: #6b7280; font-weight: 500; }
        .badge-high { background: #fee2e2; color: #991b1b; }
        .badge-medium { background: #fef3c7; color: #854d0e; }
        .badge-low { background: #dcfce7; color: #166534; }
    </style>
</head>
<body class="p-6">
    <div class="max-w-7xl mx-auto space-y-6">

        <div class="flex flex-col md:flex-row justify-between items-start md:items-end border-b border-gray-300 pb-4 gap-4">
            <div>
                <h1 class="text-3xl font-bold text-gray-900 mb-1">Enterprise Category Intelligence</h1>
                <div class="flex items-center space-x-2">
                    <span class="text-sm font-bold text-gray-500">Active Category:</span>
                    <select id="cat-selector" class="bg-white border border-gray-300 rounded p-1 text-sm font-bold text-blue-600 outline-none"></select>
                </div>
            </div>
            <div class="flex space-x-6">
                <button id="tab-risk" class="pb-2 tab-active outline-none">Friction & Metadata</button>
                <button id="tab-bench" class="pb-2 tab-inactive outline-none">Brand Strategy</button>
            </div>
        </div>

        <div id="view-risk" class="space-y-6">
            <div class="grid grid-cols-1 md:grid-cols-2 gap-6">
                <div class="card border-t-4 border-red-500">
                    <h2 class="font-bold text-gray-800 mb-4">Top B2B Friction Points</h2>
                    <ul id="friction-list" class="space-y-4 text-sm text-gray-700"></ul>
                </div>
                <div class="card border-t-4 border-green-500">
                    <h2 class="font-bold text-gray-800 mb-4">Metadata Enrichment Opportunities</h2>
                    <ul id="metadata-list" class="space-y-4 text-sm text-gray-700"></ul>
                </div>
            </div>

            <div class="card bg-gray-50">
                <div class="flex justify-between items-center mb-4">
                    <h2 class="font-bold text-gray-800">Sub-Category Deep Dive</h2>
                    <select id="subcat-select" class="w-1/3 p-2 border bg-white rounded text-sm outline-none"></select>
                </div>
                <div class="grid grid-cols-2 md:grid-cols-4 gap-4 bg-white p-4 rounded border">
                    <div><div class="text-xs font-bold text-gray-400 uppercase">Primary Driver</div><div id="detail-driver" class="text-sm font-bold text-green-700 mt-1">...</div></div>
                    <div><div class="text-xs font-bold text-gray-400 uppercase">Top Complaint</div><div id="detail-complaint" class="text-sm font-bold text-red-700 mt-1">...</div></div>
                    <div><div class="text-xs font-bold text-gray-400 uppercase">Winner Attribute</div><div id="detail-winner" class="text-sm font-bold text-blue-700 mt-1">...</div></div>
                    <div><div class="text-xs font-bold text-gray-400 uppercase">Metadata Gap</div><div id="detail-gap" class="text-sm font-bold text-purple-700 mt-1">...</div></div>
                </div>
                <div class="mt-4 p-4 border-l-4 border-yellow-400 bg-yellow-50 rounded text-sm text-gray-800">
                    <span class="font-bold text-yellow-800">SKU Specific Callout:</span> <span id="detail-sku">...</span>
                </div>
            </div>
        </div>

        <div id="view-bench" class="hidden space-y-6">
            <div class="card border-t-4 border-indigo-600 bg-indigo-50">
                <h2 class="font-bold text-indigo-900 mb-4">Top Strategic Findings</h2>
                <ul id="strategic-list" class="space-y-3 text-sm text-indigo-800"></ul>
            </div>

            <h2 class="font-bold text-xl text-gray-800 mt-8 mb-2">Brand Intelligence & Positioning</h2>
            <div id="brand-grid" class="grid grid-cols-1 md:grid-cols-2 lg:grid-cols-3 gap-6"></div>
        </div>

    </div>

    <script>
        const masterData = DATA_PLACEHOLDER;

        // Tab Management
        document.getElementById('tab-risk').onclick = (e) => {
            document.getElementById('view-risk').classList.remove('hidden');
            document.getElementById('view-bench').classList.add('hidden');
            e.target.className = "pb-2 tab-active outline-none";
            document.getElementById('tab-bench').className = "pb-2 tab-inactive outline-none";
        };
        document.getElementById('tab-bench').onclick = (e) => {
            document.getElementById('view-bench').classList.remove('hidden');
            document.getElementById('view-risk').classList.add('hidden');
            e.target.className = "pb-2 tab-active outline-none";
            document.getElementById('tab-risk').className = "pb-2 tab-inactive outline-none";
        };

        const catSelector = document.getElementById('cat-selector');
        Object.keys(masterData).forEach(cat => {
            catSelector.innerHTML += `<option value="${cat}">${cat.replace('_', ' ')}</option>`;
        });
        catSelector.onchange = (e) => renderView(e.target.value);

        function renderView(catKey) {
            const data = masterData[catKey];
            if(!data) return;

            // Render Friction List (New Schema)
            document.getElementById('friction-list').innerHTML = (data.b2b_friction_points || []).map(item => `
                <li class='flex items-start'>
                    <span class='text-red-500 mr-2'>✕</span>
                    <div>
                        <div class="font-medium">${item.issue}</div>
                        <div class="text-xs text-gray-400 mt-1">Evidence: ${item.evidence_strength.toUpperCase()}</div>
                    </div>
                </li>`).join('');

            // Render Metadata Gaps (New Schema)
            document.getElementById('metadata-list').innerHTML = (data.metadata_enrichment_opportunities || []).map(item => `
                <li class='flex items-start'>
                    <span class='text-green-500 mr-2'>+</span>
                    <div class="w-full">
                        <div class="font-bold text-gray-800">${item.attribute}</div>
                        <div class="flex justify-between text-xs text-gray-500 mt-1">
                            <span>Review Frequency: ${item.review_frequency_percent}%</span>
                            <span class="font-bold text-purple-600">Gap Score: ${item.gap_score}</span>
                        </div>
                    </div>
                </li>`).join('');

            // Subcategory Breakdowns
            const subSelect = document.getElementById('subcat-select');
            subSelect.innerHTML = '';
            (data.subcategory_breakdowns || []).forEach((sub, idx) => {
                subSelect.innerHTML += `<option value="${idx}">${sub.subcategory_name}</option>`;
            });
            subSelect.onchange = (e) => {
                const sub = data.subcategory_breakdowns[e.target.value];
                document.getElementById('detail-driver').textContent = sub?.primary_purchase_driver || 'N/A';
                document.getElementById('detail-complaint').textContent = sub?.top_complaint || 'N/A';
                document.getElementById('detail-winner').textContent = sub?.winner_attribute || 'N/A';
                document.getElementById('detail-gap').textContent = sub?.metadata_gap || 'N/A';
                document.getElementById('detail-sku').textContent = sub?.sku_specific_callout || 'N/A';
            };
            if(data.subcategory_breakdowns?.length > 0) subSelect.dispatchEvent(new Event('change'));

            // Top Strategic Findings
            document.getElementById('strategic-list').innerHTML = (data.top_strategic_findings || []).map(item => `
                <li class="p-3 bg-white rounded border border-indigo-100 shadow-sm">
                    <div class="font-bold mb-1">${item.finding}</div>
                    <div class="text-gray-600 italic">"${item.supporting_evidence}"</div>
                </li>`).join('');

            // Brand Insights Grid
            document.getElementById('brand-grid').innerHTML = (data.brand_insights || []).map(brand => `
                <div class="card border border-gray-200">
                    <h3 class="font-bold text-lg text-gray-900 mb-3 border-b pb-2">${brand.brand}</h3>
                    <div class="mb-2"><span class="text-xs font-bold text-green-600 uppercase">Strength</span><p class="text-sm text-gray-700">${brand.strength}</p></div>
                    <div class="mb-2"><span class="text-xs font-bold text-red-600 uppercase">Weakness</span><p class="text-sm text-gray-700">${brand.weakness}</p></div>
                    <div><span class="text-xs font-bold text-blue-600 uppercase">Buyer Association</span><p class="text-sm text-gray-700">${brand.buyer_association}</p></div>
                </div>
            `).join('');
        }

        if(Object.keys(masterData).length > 0) renderView(Object.keys(masterData)[0]);
    </script>
</body>
</html>
""".replace('DATA_PLACEHOLDER', json_payload)

HTML(html_content)